In [2]:
# Import necessary libraries
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


# Transactions by Channel (Online, Walk-in, etc) and Park

In this particular EDA, we want to analyze the transactions by channel and park (Sunway Lagoon VS Lost World). We would want to see if there is any particular trend, eg, more individuals are paying using online in Sunway Lagoon, etc. 

## List of variables

1. Environment: Specify whether the environment is Sunway Lagoon or Lost World
- WHEN Environment = 1 THEN 'Sunway Lagoon'
- WHEN Environment = 2 THEN 'Lost World'

2. Transaction data
- Complete_QTY: Total Visitors
- OriginalAmount * Complete_QTY: Total Revenue

3. Park: Revenue Grouping (whether you are online, walk-in, bulk buying, etc)
- We should understand the different unique values available

4. TranDate: Transaction Date
- According to our EDA, we only focus on the data from 2023 to 2024 to train our model, skipping earlier years that is affected by COVID-19.

## Loading Dataset

In [65]:
# Connect to the database
con = duckdb.connect("theme_park_data.db")

# Extract the necessary information for EDA
# Transactions data, channels (revenue grouping), and type of park/environment
# Focus from 2023 to 2024

df_channel_park = con.execute("""
SELECT
    TRIM(Park) as Grouping,
    CASE
        WHEN Environment = 1 THEN 'Sunway Lagoon'
        WHEN Environment = 2 THEN 'Lost World'
    END AS Park,
    OriginalAmount AS Unit_Price,
    Complete_QTY AS No_of_Visitors,
    (OriginalAmount * Complete_QTY) AS Total_Revenue
FROM theme_park_database
WHERE Environment IN (1, 2) 
    AND Grouping IS NOT NULL
    AND TranDate >= TIMESTAMP '2023-01-01 00:00:00'
    AND TranDate < TIMESTAMP '2025-01-01 00:00:00'
""").df()

df_channel_park.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Grouping,Park,Unit_Price,No_of_Visitors,Total_Revenue
0,Group Function,Sunway Lagoon,0.0,0.0,0.0
1,Tour / Hotel / Presold,Sunway Lagoon,0.0,0.0,0.0
2,Membership,Lost World,0.0,0.0,0.0
3,Hotspring,Lost World,12.0,1.0,12.0
4,Hotspring,Lost World,0.0,0.0,0.0


## Summary Statistics

In [28]:
# Basic Profiling & Summary Statistics

print("### Summary Statistics by Park and Channel ###")
summary_stats = df_channel_park.groupby(['Park', 'Grouping']).agg({
    'Total_Revenue': ['sum', 'mean', 'std'],
    'No_of_Visitors': ['sum', 'count'], # 'count' here tells us the number of transaction line items
    'Unit_Price': ['mean', 'max', 'min']
}).reset_index()

summary_stats

### Summary Statistics by Park and Channel ###


Park                Grouping Total_Revenue              \
                                                    sum        mean   
0      Lost World     FOC / Complimentary  0.000000e+00    0.000000   
1      Lost World          Group Function  6.486853e+06   29.166192   
2      Lost World               Hotspring  5.571817e+06    8.438873   
3      Lost World     Inter-company/Staff  3.991750e+05   37.686461   
4      Lost World              Membership  0.000000e+00    0.000000   
5      Lost World    Tours/Hotels/Presold  7.364833e+05    0.830498   
6      Lost World              Water Park  1.374938e+07   41.000209   
7   Sunway Lagoon       B2B - night event  0.000000e+00    0.000000   
8   Sunway Lagoon       B2C - night event  0.000000e+00    0.000000   
9   Sunway Lagoon              B2C Online  0.000000e+00    0.000000   
10  Sunway Lagoon                   Event  5.598771e+05  111.953035   
11  Sunway Lagoon           Explorer pass  0.000000e+00    0.000000   
12  Sunway Lagoon     FOC / Complimentary  0.000000e+00    0.000000   
13  Sunway Lagoon          Group Function  1.959638e+07   37.943726   
14  Sunway Lagoon           Inter-company  2.977939e+05   65.810815   
15  Sunway Lagoon              Night Park  2.825277e+06    9.346432   
16  Sunway Lagoon                Passport  3.960000e+02    0.000516   
17  Sunway Lagoon                   Staff  1.685008e+06   62.122411   
18  Sunway Lagoon  Tour / Hotel / Presold  1.887431e+06    1.104401   
19  Sunway Lagoon                 Walk In  4.329948e+07   86.553086   
20  Sunway Lagoon                 Walk in  0.000000e+00    0.000000   
21  Sunway Lagoon        Walk in - Online  0.000000e+00    0.000000   

              No_of_Visitors           Unit_Price                  
          std            sum    count        mean     max     min  
0    0.000000    8281.000000    21964    0.000000    0.00    0.00  
1   36.049845   95382.000000   222410   31.647267  661.11 -434.26  
2   18.092995  325000.000000   660256    8.148516  115.00 -115.00  
3   38.257165    5032.000000    10592   35.972677   81.00  -76.80  
4    0.000000  504219.000000   605074    0.000000    0.00    0.00  
5    7.708353  476025.500000   886797    0.873190   88.00  -80.00  
6   48.081650  170751.000000   335349   40.224641  135.00 -135.00  
7    0.000000   40153.000000    40153    0.000000    0.00    0.00  
8    0.000000   64836.000000    64836    0.000000    0.00    0.00  
9    0.000000   42074.000000    42074    0.000000    0.00    0.00  
10  32.092857    4768.000000     5001  108.126907  121.90 -121.90  
11   0.000000   15122.000000    35852    0.000000    0.00    0.00  
12   0.000000   14087.833333    45180    0.000000    0.00    0.00  
13  52.774257  264575.000000   516459   33.465584  209.91 -209.91  
14  66.136928    2260.000000     4525   65.728522  137.14 -128.57  
15  20.980128  136438.000000   302284   13.122598   83.81  -83.81  
16   0.319657  681101.000000   767346    0.000258  198.00 -198.00  
17  62.328877   13171.000000    27124   60.297903  128.57 -128.57  
18  11.269758  658536.000000  1709009   18.932922  202.86 -143.81  
19  91.329632  246241.000000   500265   85.739960  238.10 -238.10  
20   0.000000     192.000000      192    0.000000    0.00    0.00  
21   0.000000   10138.000000    11138    0.000000    0.00    0.00

In [30]:
# Interested to know the unique channels (groupings) for each park
# print("### Unique Channels (Groupings) for Each Park ###")
# df_channel_park['Grouping'].value_counts()

# Based on the grouping above, we can see that there are some channels that are unique to each park, while some channels are common to both parks. For example, "Online" and "Walk-in" channels are common to both parks, while "Travel Agent" is unique to Sunway Lagoon and "Corporate" is unique to Lost World. This information can be useful for further analysis and understanding the revenue distribution across different channels for each park.
# Within each park, we can see that Walk In in repeated twice in Sunway Lagoon, let's combine them

# Clean the 'Grouping' column directly in the dataframe
# .strip() removes spaces, .title() converts 'walk in' to 'Walk In'
df_channel_park['Grouping'] = df_channel_park['Grouping'].str.strip().str.title()


# Re-run summary statistics
# Now that 'walk in' and 'Walk In' are identical, groupby will merge them

print("### Summary Statistics by Park and Channel ###")
summary_stats = df_channel_park.groupby(['Park', 'Grouping']).agg({
    'Total_Revenue': ['sum', 'mean', 'std'],
    'No_of_Visitors': ['sum', 'count'], # 'count' here tells us the number of transaction line items
    'Unit_Price': ['mean', 'max', 'min']
}).reset_index()

summary_stats

### Summary Statistics by Park and Channel ###


Park                Grouping Total_Revenue              \
                                                    sum        mean   
0      Lost World     Foc / Complimentary  0.000000e+00    0.000000   
1      Lost World          Group Function  6.486853e+06   29.166192   
2      Lost World               Hotspring  5.571817e+06    8.438873   
3      Lost World     Inter-Company/Staff  3.991750e+05   37.686461   
4      Lost World              Membership  0.000000e+00    0.000000   
5      Lost World    Tours/Hotels/Presold  7.364833e+05    0.830498   
6      Lost World              Water Park  1.374938e+07   41.000209   
7   Sunway Lagoon       B2B - Night Event  0.000000e+00    0.000000   
8   Sunway Lagoon       B2C - Night Event  0.000000e+00    0.000000   
9   Sunway Lagoon              B2C Online  0.000000e+00    0.000000   
10  Sunway Lagoon                   Event  5.598771e+05  111.953035   
11  Sunway Lagoon           Explorer Pass  0.000000e+00    0.000000   
12  Sunway Lagoon     Foc / Complimentary  0.000000e+00    0.000000   
13  Sunway Lagoon          Group Function  1.959638e+07   37.943726   
14  Sunway Lagoon           Inter-Company  2.977939e+05   65.810815   
15  Sunway Lagoon              Night Park  2.825277e+06    9.346432   
16  Sunway Lagoon                Passport  3.960000e+02    0.000516   
17  Sunway Lagoon                   Staff  1.685008e+06   62.122411   
18  Sunway Lagoon  Tour / Hotel / Presold  1.887431e+06    1.104401   
19  Sunway Lagoon                 Walk In  4.329948e+07   86.519880   
20  Sunway Lagoon        Walk In - Online  0.000000e+00    0.000000   

              No_of_Visitors           Unit_Price                  
          std            sum    count        mean     max     min  
0    0.000000    8281.000000    21964    0.000000    0.00    0.00  
1   36.049845   95382.000000   222410   31.647267  661.11 -434.26  
2   18.092995  325000.000000   660256    8.148516  115.00 -115.00  
3   38.257165    5032.000000    10592   35.972677   81.00  -76.80  
4    0.000000  504219.000000   605074    0.000000    0.00    0.00  
5    7.708353  476025.500000   886797    0.873190   88.00  -80.00  
6   48.081650  170751.000000   335349   40.224641  135.00 -135.00  
7    0.000000   40153.000000    40153    0.000000    0.00    0.00  
8    0.000000   64836.000000    64836    0.000000    0.00    0.00  
9    0.000000   42074.000000    42074    0.000000    0.00    0.00  
10  32.092857    4768.000000     5001  108.126907  121.90 -121.90  
11   0.000000   15122.000000    35852    0.000000    0.00    0.00  
12   0.000000   14087.833333    45180    0.000000    0.00    0.00  
13  52.774257  264575.000000   516459   33.465584  209.91 -209.91  
14  66.136928    2260.000000     4525   65.728522  137.14 -128.57  
15  20.980128  136438.000000   302284   13.122598   83.81  -83.81  
16   0.319657  681101.000000   767346    0.000258  198.00 -198.00  
17  62.328877   13171.000000    27124   60.297903  128.57 -128.57  
18  11.269758  658536.000000  1709009   18.932922  202.86 -143.81  
19  91.327841  246433.000000   500457   85.707066  238.10 -238.10  
20   0.000000   10138.000000    11138    0.000000    0.00    0.00

### Observation from the Summary Statistics

1. The "Ghost" Revenue Segments (Membership & B2C Online)
- Several high-volume segments show exactly 0.00 revenue despite having massive visitor counts.
- Observation: Lost World Membership (504k visitors) and Sunway Passport (681k visitors) have effectively zero revenue in this table.
- Insights: The revenue was likely collected upfront (when the pass was bought) and these rows only track the physical entry.

2. High-Skew "Tour & Hotel" Yields (High revenue, but very low unit_price)
- The Tour / Hotel / Presold category for Sunway Lagoon (Row 18) shows a massive volume but a tiny average price.
- Observation: 658k visitors, but a Mean Unit Price of only $18.93$. However, the Max Unit Price is $202.86$.
- Insight: The "Mean" is being heavily dragged down by a vast majority of transactions that are likely priced at $0$ or near-zero (possibly part of a hotel package where the room gets the revenue and the park entry is "free"). This segment is not a "standard" ticket sale; it's a bulk/package-heavy segment.

3. Price Integrity & Negative Anomalies
- Looking at the Unit_Price min column, almost every paid category has perfectly mirrored negative values (e.g., Walk In is Max $238.10$ and Min $-238.10$).
- Insight: This confirms that the system handles Refunds/Reversals by creating an identical transaction with a negative sign. This connects to the second part where we inspect the instances when the price is negative sign.
- I suspect that that might be a typo. There should be a separate record if that is consiered reimbursement, or we can even simplify our analysis by looking only at the positive revenue we collected.

4. The "Walk In" Powerhouse (Sunway Lagoon)
- Observation: Walk In (Row 19) is your "purest" and most profitable segment. It has the highest Total Revenue ($4.32 \times 10^7$) and a very healthy Mean Unit Price of $85.74$.
- Comparison: Notice that Walk In - Online (Row 20) has $10k$ visitors but $0$ revenue. This is a major anomaly. I suspect that the revenue recorded here is only about the revenue we collected at entry, so there are 2 methods we can deal with them.
- (1) Assuming the ticket price is the same from buying in-person and online, we can compute the revenue for the Walk-In Online
- (2) We can just compare the number of individuals we attracts, since the price of the ticket has been the same, so we can isolate that factor out. However, this might not be applicable to the group function, since they have significantly larger group members but getting at a very cheap per-unit price. 

5. Park Comparison: 
- "Group Functions"Observation: Sunway Lagoon (Row 13) has a Mean Unit Price for Group Functions of $33.46$, while Lost World (Row 1) is $31.64$.
- Insight: The pricing for corporate groups and events is remarkably consistent between the two parks, suggesting a centralized sales strategy or price list for B2B events.

### Questions I have when analyzing

1. How should we deal with negative unit price and number of visitors

2. In the grouping where the revenue = 0, is it still informative to compare total revenue? If we compare total number of visitors, how could we deal with the situation of complementary/FOC or even group function that contributes a large number of visitors but not as high revenue? --> Might be able to solve it using Yield/Revenue

3. This analysis is only focusing on entry ticket revenue, excluding other factors such as f&b, souvenirs and other expenses spent in the park.

In [33]:
# Rename columns for clarity
summary_stats.columns = ['Park', 'Channel', 'Total Revenue', 'Avg Trans Value', 'Revenue StdDev', 
                        'Total Visitors', 'Trans Count', 'Avg Ticket Price', 'Max Ticket Price', 'Min Ticket Price']

# 3. Calculate "Yield per Visitor" (Key Theme Park Metric)
summary_stats['Yield_Per_Visitor'] = summary_stats['Total Revenue'] / summary_stats['Total Visitors']

summary_stats.sort_values(by=['Park', 'Total Revenue'], ascending=[True, False])
summary_stats

,Park,Channel,Total Revenue,Avg Trans Value,Revenue StdDev,Total Visitors,Trans Count,Avg Ticket Price,Max Ticket Price,Min Ticket Price,Yield_Per_Visitor
0,Lost World,Foc / Complimentary,0.000000e+00,0.000000,0.000000,8281.000000,21964,0.000000,0.00,0.00,0.000000
1,Lost World,Group Function,6.486853e+06,29.166192,36.049845,95382.000000,222410,31.647267,661.11,-434.26,68.009193
2,Lost World,Hotspring,5.571817e+06,8.438873,18.092995,325000.000000,660256,8.148516,115.00,-115.00,17.144051
3,Lost World,Inter-Company/Staff,3.991750e+05,37.686461,38.257165,5032.000000,10592,35.972677,81.00,-76.80,79.327305
4,Lost World,Membership,0.000000e+00,0.000000,0.000000,504219.000000,605074,0.000000,0.00,0.00,0.000000
5,Lost World,Tours/Hotels/Presold,7.364833e+05,0.830498,7.708353,476025.500000,886797,0.873190,88.00,-80.00,1.547151
6,Lost World,Water Park,1.374938e+07,41.000209,48.081650,170751.000000,335349,40.224641,135.00,-135.00,80.522978
7,Sunway Lagoon,B2B - Night Event,0.000000e+00,0.000000,0.000000,40153.000000,40153,0.000000,0.00,0.00,0.000000
8,Sunway Lagoon,B2C - Night Event,0.000000e+00,0.000000,0.000000,64836.000000,64836,0.000000,0.00,0.00,0.000000
9,Sunway Lagoon,B2C Online,0.000000e+00,0.000000,0.000000,42074.000000,42074,0.000000,0.00,0.00,0.000000


In [ ]:
# Identifying Anomalies
print("\n### ANOMALY DETECTION ###")

# Anomaly A: Zero Revenue with High Visitors
zero_rev = summary_stats[summary_stats['Total Revenue'] == 0]
print(f"\n1. Channels with 0 Revenue but >0 Visitors: {len(zero_rev)}")
print(zero_rev[['Park', 'Channel', 'Total Visitors']])


### ANOMALY DETECTION ###

1. Channels with 0 Revenue but >0 Visitors: 8
             Park              Channel  Total Visitors
0      Lost World  Foc / Complimentary     8281.000000
4      Lost World           Membership   504219.000000
7   Sunway Lagoon    B2B - Night Event    40153.000000
8   Sunway Lagoon    B2C - Night Event    64836.000000
9   Sunway Lagoon           B2C Online    42074.000000
11  Sunway Lagoon        Explorer Pass    15122.000000
12  Sunway Lagoon  Foc / Complimentary    14087.833333
20  Sunway Lagoon     Walk In - Online    10138.000000


In [52]:
# Anomaly B: Negative values (Refunds or Data errors)
neg_ticket = df_channel_park[df_channel_park['Unit_Price'] < 0]
if not neg_ticket.empty:
    print(f"\n2. Detected {len(neg_ticket)} instances of negative revenue (potential refunds).")


2. Detected 41104 instances of negative revenue (potential refunds).


In [54]:
# Inspecting the dataframe with ticket price <0
df_channel_park[df_channel_park['Unit_Price'] < 0].head(10)

,Grouping,Park,Unit_Price,No_of_Visitors,Total_Revenue
84,Group Function,Lost World,-50.00,-1.00,50.00
139,Walk In,Sunway Lagoon,-180.95,0.00,-0.00
233,Walk In,Sunway Lagoon,-209.52,0.00,-0.00
657,Group Function,Lost World,-57.00,-0.25,14.25
731,Group Function,Sunway Lagoon,-115.24,0.00,-0.00
877,Water Park,Lost World,-0.01,0.00,-0.00
1027,Group Function,Sunway Lagoon,-80.00,0.00,-0.00
1063,Group Function,Sunway Lagoon,-109.52,0.00,-0.00
1093,Group Function,Sunway Lagoon,-104.76,0.00,-0.00
1392,Hotspring,Lost World,-12.00,-1.00,12.00


## 🔴 Question: Negative Unit_Price and No_of_Visitors
When inspecting the instances when the unit price <0, as you can see above, we do have negative number of visitors and even fraction of them. This is concerning, especially we get a positive total revenue, when in fact, our unit price and the number of visitors is in negative values. 

Guessing:
1. Is the Unit Price supposed to be in positive, but this is just the case that there is a typo? Or it is mean to be a refund?
2. In either of the case above, if there is a refund/revenue, it does not make sense for the number of visitors to be 0, fraction <1 or even negative. This makes me want to check how many of the visitors are in fraction. 

In [58]:
# Inspecting the dataframe with ticket price <0
df_fraction_visitor = df_channel_park[(df_channel_park['No_of_Visitors'] > 0) & (df_channel_park['No_of_Visitors'] < 1)]

print("### Summary Statistics by Park and Channel ###")
summary_stats = df_fraction_visitor.groupby(['Park', 'Grouping']).agg({
    'Total_Revenue': ['sum', 'mean', 'std'],
    'No_of_Visitors': ['sum', 'count'], # 'count' here tells us the number of transaction line items
    'Unit_Price': ['mean', 'max', 'min']
}).reset_index()

summary_stats

### Summary Statistics by Park and Channel ###


Park                Grouping  Total_Revenue                        \
                                                    sum       mean        std   
0     Lost World          Group Function  493414.401167  12.540077  14.115427   
1     Lost World               Hotspring       0.000000   0.000000   0.000000   
2     Lost World    Tours/Hotels/Presold   17030.437000   5.288956   6.848578   
3     Lost World              Water Park       0.000000   0.000000   0.000000   
4  Sunway Lagoon     FOC / Complimentary       0.000000   0.000000   0.000000   
5  Sunway Lagoon              Night Park       0.000000   0.000000   0.000000   
6  Sunway Lagoon  Tour / Hotel / Presold    2676.220000   0.817416   8.502138   
7  Sunway Lagoon                 Walk In  141414.910000  38.323824  20.277205   

  No_of_Visitors        Unit_Price                 
             sum  count       mean     max    min  
0   14053.700000  39347  34.684641  661.11   0.00  
1    1127.000000   2254   0.000000    0.00   0.00  
2     745.250000   3220  22.662693   80.00   0.50  
3    3402.000000   6804   0.000000    0.00   0.00  
4     946.833333   3123   0.000000    0.00   0.00  
5     785.000000   1570   0.000000    0.00   0.00  
6    1637.000000   3274   1.634832  179.05   0.00  
7    1845.000000   3690  76.647648  138.10  20.11

In [61]:
df_fraction_visitor.head(10)

,Grouping,Park,Unit_Price,No_of_Visitors,Total_Revenue
218,Group Function,Lost World,60.00,0.50,30.0000
310,Night Park,Sunway Lagoon,0.00,0.50,0.0000
324,Group Function,Lost World,148.21,0.25,37.0525
331,Group Function,Lost World,60.00,0.50,30.0000
524,Group Function,Lost World,13.00,0.50,6.5000
749,Group Function,Lost World,0.00,0.20,0.0000
774,Tour / Hotel / Presold,Sunway Lagoon,0.00,0.50,0.0000
781,Group Function,Lost World,13.00,0.50,6.5000
792,Group Function,Lost World,2.00,0.25,0.5000
844,Walk In,Sunway Lagoon,40.00,0.50,20.0000


In [60]:
df_fraction_visitor.shape[0]/df_channel_park.shape[0]

0.009287219616429993

While this might not solve the problem, we can temporarily drop this columns, because it is only 0.92% of the whole dataset we have. We might need to clarify with the data team in terms of the meaning of the fraction of the total number of visitors. 

### 🟡 Dirty Data Processing 

Hence, here are the steps I would do to temporary "correct" the dataset before heading into generating and comparing the transactions. 
1. Correcting the sign of the ticket price (I suspect that is a typo of negative sign, and I believe that there should be a separate section by the data regarding reimbursement column)
2. Correcting the sign of the number of visitors (because number of visitor is never negative).
3. Dropping all the number of visitors which is <1, to ensure our result is still interpretable and not having weird visitor number. 

In [66]:
# 1. Correcting the sign of the ticket price (OriginalAmount)
# Using absolute value to ensure all prices are positive
df_channel_park['Unit_Price'] = df_channel_park['Unit_Price'].abs()

# 2. Correcting the sign of the number of visitors (Complete_QTY)
# Ensuring that negative visitor counts (likely reversal logs) are treated as positive volume
df_channel_park['No_of_Visitors'] = df_channel_park['No_of_Visitors'].abs()

# 3. Drop fractional visitors but keep whole numbers (including 0)
# This keeps 0.0, 1.0, 2.0 etc., and removes 0.5, 1.7, etc.
initial_count = len(df_channel_park)
df_channel_park = df_channel_park[df_channel_park['No_of_Visitors'] % 1 == 0]

# Recalculate total_revenue to ensure it matches the corrected signs
df_channel_park['Total_Revenue'] = df_channel_park['Unit_Price'] * df_channel_park['No_of_Visitors']

dropped_count = initial_count - len(df_channel_park)

print(f"Data Cleaning Complete:")
print(f"- Fixed negative signs for Prices and Visitor counts.")
print(f"- Dropped {dropped_count} rows with visitor counts < 1.")
print(f"- New dataframe shape: {df_channel_park.shape}")
print(f"Amount of data lost: {dropped_count} rows ({(dropped_count/initial_count)*100:.2f}%)")

print("### Summary Statistics by Park and Channel ###")
summary_stats = df_channel_park.groupby(['Park', 'Grouping']).agg({
    'Total_Revenue': ['sum', 'mean', 'std'],
    'No_of_Visitors': ['sum', 'count'], # 'count' here tells us the number of transaction line items
    'Unit_Price': ['mean', 'max', 'min']
}).reset_index()

summary_stats

Data Cleaning Complete:
- Fixed negative signs for Prices and Visitor counts.
- Dropped 64100 rows with visitor counts < 1.
- New dataframe shape: (6749780, 5)
Amount of data lost: 64100 rows (0.94%)
### Summary Statistics by Park and Channel ###


Park                Grouping Total_Revenue              \
                                                    sum        mean   
0      Lost World     FOC / Complimentary          0.00    0.000000   
1      Lost World          Group Function    5987376.97   32.843538   
2      Lost World               Hotspring    5571816.55    8.468038   
3      Lost World     Inter-company/Staff     399175.00   37.686461   
4      Lost World              Membership          0.00    0.000000   
5      Lost World    Tours/Hotels/Presold     719178.00    0.813968   
6      Lost World              Water Park   13749379.00   41.849812   
7   Sunway Lagoon       B2B - night event          0.00    0.000000   
8   Sunway Lagoon       B2C - night event          0.00    0.000000   
9   Sunway Lagoon              B2C Online          0.00    0.000000   
10  Sunway Lagoon                   Event     559877.13  111.953035   
11  Sunway Lagoon           Explorer pass          0.00    0.000000   
12  Sunway Lagoon     FOC / Complimentary          0.00    0.000000   
13  Sunway Lagoon          Group Function   19596379.03   37.943726   
14  Sunway Lagoon           Inter-company     297793.94   65.810815   
15  Sunway Lagoon              Night Park    2825276.83    9.395229   
16  Sunway Lagoon                Passport        396.00    0.000516   
17  Sunway Lagoon                   Staff    1685008.28   62.122411   
18  Sunway Lagoon  Tour / Hotel / Presold    1884754.50    1.104952   
19  Sunway Lagoon                 Walk In   43158064.65   86.911473   
20  Sunway Lagoon                 Walk in          0.00    0.000000   
21  Sunway Lagoon        Walk in - Online          0.00    0.000000   

              No_of_Visitors           Unit_Price               
          std            sum    count        mean     max  min  
0    0.000000        11253.0    21964    0.000000    0.00  0.0  
1   38.301876        85820.0   182300   32.950504  250.00  0.0  
2   18.117419       334091.0   657982    8.468038  115.00  0.0  
3   38.257165         5308.0    10592   37.686461   81.00  0.0  
4    0.000000       505159.0   605074    0.000000    0.00  0.0  
5    7.706429       480642.0   883546    0.813968   88.00  0.0  
6   48.209908       170051.0   328541   41.850498  135.00  0.0  
7    0.000000        40153.0    40153    0.000000    0.00  0.0  
8    0.000000        64836.0    64836    0.000000    0.00  0.0  
9    0.000000        42074.0    42074    0.000000    0.00  0.0  
10  32.092857         4768.0     5001  117.136941  121.90  0.0  
11   0.000000        15122.0    35852    0.000000    0.00  0.0  
12   0.000000        13141.0    42057    0.000000    0.00  0.0  
13  52.774257       264575.0   516459   42.421869  209.91  0.0  
14  66.136928         2260.0     4525   65.893109  137.14  0.0  
15  21.023924       135653.0   300714   13.804175   83.81  0.0  
16   0.319657       681101.0   767346    0.000774  198.00  0.0  
17  62.328877        13567.0    27124   62.134349  128.57  0.0  
18  11.274412       656899.0  1705735   18.971695  202.86  0.0  
19  91.556627       244396.0   496575   88.015744  238.10  0.0  
20   0.000000          192.0      192    0.000000    0.00  0.0  
21   0.000000        10138.0    11138    0.000000    0.00  0.0

## Visual Summary

To better visualize the relationship between transactions with the channels and the parks, we can plot side-by-side boxplot to see the distribution easily across two different parks. 

In particular I would want to address the following questions via visualization:

1. Bar charts showing the total number of visitors VS revenue that is generated by different grouping for different parks. 

2. Yield per visitors comparison between the 2 parks. 

3. The yield per visitors according to different channels and parks. 

4. Realizing that there is a lot of zero revenue but large number of individuals, I want to see how much percentage of the park is occupied by the free-entry individuals compared to those who pays (Visitor Mix Chart)